#02 — Data Preprocessing & Preparation 

In [ ]:
import pandas as pd
from pathlib import Path

# Lokasi utama proyek Datavidia
PROJECT_DIR = Path.cwd().parent.parent

# Lokasi dataset asli
DATASET_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "task-1-datavidia-11-0"
    / "Task-1-Datavidia-11.0"
)

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

# Membaca dataset utama
train = pd.read_csv(TRAIN_DIR / "shipments.csv", low_memory=False)
test = pd.read_csv(TEST_DIR / "shipments.csv", low_memory=False)

TARGET = "status_penanganan"

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nKolom train:")
print(train.columns.tolist())

print("\nKolom test:")
print(test.columns.tolist())

print("\nDistribusi target:")
print(train[TARGET].value_counts())

In [ ]:
FEATURES_DIR = Path.cwd().parent / "data"
inv_feat = pd.read_csv(FEATURES_DIR / "inv_feat.csv")
route_feat = pd.read_csv(FEATURES_DIR / "route_feat.csv")
scan_feat = pd.read_csv(FEATURES_DIR / "scan_feat.csv")

# Memeriksa kualitas kunci penggabungan
datasets = {
    "Train": train,
    "Test": test,
    "Invoice": inv_feat,
    "Route": route_feat,
    "Scan": scan_feat
}

for nama, df in datasets.items():
    print("\n" + "=" * 45)
    print(nama)
    print("Jumlah baris:", len(df))
    print("Jumlah shipment_id kosong:", df["shipment_id"].isna().sum())
    print("Jumlah shipment_id duplikat:", df["shipment_id"].duplicated().sum())
    print("Jumlah shipment_id unik:", df["shipment_id"].nunique())

In [ ]:
feature_datasets = {
    "Invoice": inv_feat,
    "Route": route_feat,
    "Scan": scan_feat
}

for nama, fitur in feature_datasets.items():
    print("\n" + "=" * 45)
    print(nama)

    print(
        "ID fitur yang cocok dengan train:",
        fitur["shipment_id"].isin(train["shipment_id"]).sum()
    )

    print(
        "ID fitur yang cocok dengan test:",
        fitur["shipment_id"].isin(test["shipment_id"]).sum()
    )

    print(
        "ID fitur yang tidak ditemukan di train:",
        (~fitur["shipment_id"].isin(train["shipment_id"])).sum()
    )

In [ ]:
train_pitri = train.copy()

train_pitri = train_pitri.merge(
    inv_feat,
    on="shipment_id",
    how="left",
    validate="one_to_one"
)

train_pitri = train_pitri.merge(
    route_feat,
    on="shipment_id",
    how="left",
    validate="one_to_one"
)

train_pitri = train_pitri.merge(
    scan_feat,
    on="shipment_id",
    how="left",
    validate="one_to_one"
)

print("Train sebelum merge:", train.shape)
print("Train setelah merge:", train_pitri.shape)

print("\nJumlah missing value fitur baru:")
print(
    train_pitri[
        list(inv_feat.columns[1:])
        + list(route_feat.columns[1:])
        + list(scan_feat.columns[1:])
    ].isna().sum()
)

In [ ]:
invoices_test = pd.read_csv(TEST_DIR / "invoices.csv", low_memory=False)
routes_test = pd.read_csv(TEST_DIR / "route_segments.csv", low_memory=False)
scans_test = pd.read_csv(TEST_DIR / "scan_events.csv", low_memory=False)

print("INVOICES TEST:", invoices_test.shape)
print("ROUTE SEGMENTS TEST:", routes_test.shape)
print("SCAN EVENTS TEST:", scans_test.shape)

In [ ]:
raw_test_datasets = {
    "Invoices Test": invoices_test,
    "Route Segments Test": routes_test,
    "Scan Events Test": scans_test
}

for nama, df in raw_test_datasets.items():
    print("\n" + "=" * 60)
    print(nama)
    print("Kolom:")
    print(df.columns.tolist())

    print("\nLima baris pertama:")
    display(df.head())

In [ ]:
print("KOLOM ROUTE:")
print(routes_test.columns.tolist())

display(routes_test.head(5))

In [ ]:
print("KOLOM SCAN:")
print(scans_test.columns.tolist())

display(scans_test.head(5))

In [ ]:
print("KOLOM INVOICE:")
print(invoices_test.dtypes)

display(invoices_test.head(3))

In [ ]:
%pip install polars

In [ ]:

import sys
import gc
import types
import importlib.util
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

# Lokasi proyek
PROJECT_DIR = Path.cwd().parent.parent

DATASET_DIR = (
    PROJECT_DIR / "data" / "raw" /
    "task-1-datavidia-11-0" /
    "Task-1-Datavidia-11.0"
)

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

PITRI_DIR = PROJECT_DIR / "pitri"
PROCESSED_DIR = PITRI_DIR / "data" / "processed"
MODEL_DIR = PITRI_DIR / "models"
RESULT_DIR = PITRI_DIR / "results"

for folder in [PROCESSED_DIR, MODEL_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Import pipeline Farhat
SRC_DIR = PROJECT_DIR / "farhat" / "src"

package = types.ModuleType("farhat_pkg")
package.__path__ = [str(SRC_DIR)]
sys.modules["farhat_pkg"] = package

spec = importlib.util.spec_from_file_location(
    "farhat_pkg.data_loader",
    SRC_DIR / "data_loader.py"
)
data_loader = importlib.util.module_from_spec(spec)
sys.modules["farhat_pkg.data_loader"] = data_loader
spec.loader.exec_module(data_loader)

# Arahkan ke dataset utama
data_loader.RAW_DIR = DATASET_DIR
data_loader.TRAIN_DIR = TRAIN_DIR
data_loader.TEST_DIR = TEST_DIR
data_loader.PROCESSED_DIR = PROCESSED_DIR

spec = importlib.util.spec_from_file_location(
    "farhat_pkg.features",
    SRC_DIR / "features.py"
)
features = importlib.util.module_from_spec(spec)
sys.modules["farhat_pkg.features"] = features
spec.loader.exec_module(features)

features.PROCESSED_DIR = PROCESSED_DIR

print("Konfigurasi berhasil.")
print("Train tersedia:", TRAIN_DIR.exists())
print("Test tersedia:", TEST_DIR.exists())
print("Folder output:", PROCESSED_DIR)


In [ ]:

import gc
from pathlib import Path

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Membangun fitur train...")
hasil = features.build_features("train")
del hasil
gc.collect()

print("Membangun fitur test...")
hasil = features.build_features("test")
del hasil
gc.collect()

print("Selesai membangun fitur.")
print("Train:", PROCESSED_DIR / "feat_train.parquet")
print("Test :", PROCESSED_DIR / "feat_test.parquet")


In [ ]:

import polars as pl

train_path = PROCESSED_DIR / "feat_train.parquet"
test_path = PROCESSED_DIR / "feat_test.parquet"

tr = pl.scan_parquet(train_path)
te = pl.scan_parquet(test_path)

print("Train:", tr.select(pl.len()).collect().item(), "baris")
print("Test :", te.select(pl.len()).collect().item(), "baris")
print("Kolom train:", len(tr.collect_schema().names()))
print("Kolom test:", len(te.collect_schema().names()))


In [ ]:
%pip install pyarrow

In [ ]:
import pyarrow

print("PyArrow berhasil:", pyarrow.__version__)

In [ ]:

import gc
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from lightgbm import LGBMClassifier

gc.collect()

PROJECT_DIR = Path.cwd().parent.parent
DATASET_DIR = PROJECT_DIR / "data" / "raw" / "task-1-datavidia-11-0" / "Task-1-Datavidia-11.0"
PROCESSED_DIR = PROJECT_DIR / "pitri" / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "pitri" / "models"
RESULT_DIR = PROJECT_DIR / "pitri" / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "status_penanganan"
ID_COL = "shipment_id"

train_path = PROCESSED_DIR / "feat_train.parquet"
test_path = PROCESSED_DIR / "feat_test.parquet"

train_lazy = pl.scan_parquet(train_path)
test_lazy = pl.scan_parquet(test_path)

train_schema = train_lazy.collect_schema()
test_schema = test_lazy.collect_schema()

feature_cols = [
    c for c in train_schema.names()
    if c != TARGET and not c.endswith("_id")
]

missing_cols = [c for c in feature_cols if c not in test_schema.names()]
if missing_cols:
    raise ValueError(f"Kolom test tidak lengkap: {missing_cols}")

cat_cols = [
    c for c in feature_cols
    if str(train_schema[c]) in ("String", "Utf8", "Categorical", "Object")
    or str(train_schema[c]).startswith("Enum")
]

num_cols = [c for c in feature_cols if c not in cat_cols]
cat_indices = [feature_cols.index(c) for c in cat_cols]

print("Mengambil sampel training...")

train_sample = (
    train_lazy
    .filter((pl.col(ID_COL).hash(seed=42) % 50) == 0)
    .select([TARGET] + feature_cols)
    .collect()
)

if train_sample.height == 0:
    raise ValueError("Sampel training kosong.")

y_train = np.asarray(
    train_sample[TARGET].cast(pl.String).to_list(),
    dtype=str
)

cat_maps = {}

for c in cat_cols:
    values = (
        train_sample[c]
        .cast(pl.String)
        .fill_null("MISSING")
        .unique()
        .to_list()
    )
    cat_maps[c] = {v: i for i, v in enumerate(values)}

expressions = []

for c in cat_cols:
    expressions.append(
        pl.col(c)
        .cast(pl.String)
        .fill_null("MISSING")
        .replace_strict(
            cat_maps[c],
            default=-1,
            return_dtype=pl.Int32
        )
        .alias(c)
    )

for c in num_cols:
    numeric = pl.col(c).cast(pl.Float32, strict=False)
    expressions.append(
        pl.when(numeric.is_finite())
        .then(numeric)
        .otherwise(None)
        .alias(c)
    )

train_sample = train_sample.with_columns(expressions)

X_train = (
    train_sample
    .select(feature_cols)
    .to_numpy()
    .astype(np.float32, copy=False)
)

print("Sampel:", X_train.shape)
print("Jumlah kelas:", len(np.unique(y_train)))

del train_sample
gc.collect()

model = LGBMClassifier(
    objective="multiclass",
    n_estimators=250,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=7,
    max_bin=31,
    min_child_samples=50,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=1,
    verbosity=-1,
    force_col_wise=True
)

print("Mulai training...")

model.fit(
    X_train,
    y_train,
    categorical_feature=cat_indices
)

model.booster_.save_model(
    str(MODEL_DIR / "lightgbm_final.txt")
)

print("Training selesai.")

del X_train, y_train
gc.collect()

test_count = test_lazy.select(pl.len()).collect().item()
test_ids = []
predictions = []

batch_size = 5000

print("Mulai prediksi test...")

for start in range(0, test_count, batch_size):
    batch = (
        test_lazy
        .slice(start, batch_size)
        .select([ID_COL] + feature_cols)
        .collect()
    )

    test_ids.extend(
        batch[ID_COL].cast(pl.String).to_list()
    )

    batch = batch.with_columns(expressions)

    X_batch = (
        batch
        .select(feature_cols)
        .to_numpy()
        .astype(np.float32, copy=False)
    )

    batch_pred = model.predict(X_batch)
    predictions.extend(batch_pred.tolist())

    del batch, X_batch, batch_pred
    gc.collect()

    print(f"{min(start + batch_size, test_count)} / {test_count}")

sample = pd.read_csv(DATASET_DIR / "sample_submission.csv")

if ID_COL not in sample.columns:
    raise ValueError(f"Kolom ID tidak ditemukan: {sample.columns.tolist()}")

if len(predictions) != len(test_ids):
    raise ValueError("Jumlah prediksi dan ID tidak sama.")

target_cols = [c for c in sample.columns if c != ID_COL]

target_col = TARGET if TARGET in target_cols else (
    target_cols[0] if len(target_cols) == 1 else None
)

if target_col is None:
    raise ValueError(f"Kolom target sample tidak jelas: {sample.columns.tolist()}")

pred_df = pd.DataFrame({
    ID_COL: test_ids,
    "_prediction": predictions
})

sample[ID_COL] = sample[ID_COL].astype(str)

submission = sample.merge(
    pred_df,
    on=ID_COL,
    how="left",
    sort=False,
    validate="one_to_one"
)

if submission["_prediction"].isna().any():
    raise ValueError("Ada ID yang belum mendapatkan prediksi.")

submission[target_col] = submission["_prediction"]
submission = submission.drop(columns="_prediction")
submission = submission[sample.columns]

submission_path = RESULT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("\nSUBMISSION BERHASIL DIBUAT")
print("Lokasi:", submission_path)
print("Ukuran:", submission.shape)
print(submission.head())
print(submission[target_col].value_counts())


In [ ]:

from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent.parent
PITRI_DIR = ROOT / "pitri"

TRAIN_PATH = PITRI_DIR / "data" / "processed" / "feat_train.parquet"
TEST_PATH = PITRI_DIR / "data" / "processed" / "feat_test.parquet"

train = pl.scan_parquet(TRAIN_PATH)
test = pl.scan_parquet(TEST_PATH)

train_schema = train.collect_schema()
test_schema = test.collect_schema()

print("TRAIN:", train.select(pl.len()).collect().item())
print("TEST :", test.select(pl.len()).collect().item())

print("\nJumlah fitur train:", len(train_schema.names()) - 1)
print("Jumlah fitur test :", len(test_schema.names()))

print("\nDistribusi target:")
print(
    train.group_by("status_penanganan")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .collect()
)

print("\nTipe data fitur:")
for col, dtype in train_schema.items():
    print(f"{col}: {dtype}")

print("\nFile submission sebelumnya:")
print(PITRI_DIR / "results" / "submission.csv")


In [ ]:

import polars as pl
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

print("Menyiapkan sampel data...")

schema = train.collect_schema()

target = "status_penanganan"
id_col = "shipment_id"

fitur = [
    c for c in schema.names()
    if c not in [target, id_col]
]

kategori = [
    c for c in fitur
    if schema[c] in [pl.String, pl.Categorical, pl.Enum]
]

numerik = [c for c in fitur if c not in kategori]

data = (
    pl.scan_parquet(TRAIN_PATH)
    .filter(pl.col(id_col).hash(seed=42) % 50 == 0)
    .select([target] + numerik + kategori)
    .collect()
)

print("Ukuran sampel:", data.shape)

print("Mengubah fitur kategorikal...")

data = data.with_columns([
    pl.col(c)
    .cast(pl.String)
    .fill_null("__MISSING__")
    .cast(pl.Categorical)
    .to_physical()
    .cast(pl.Float32)
    .alias(c)
    for c in kategori
])

data = data.with_columns([
    pl.col(c).cast(pl.Float32).alias(c)
    for c in numerik
])

X = data.select(fitur).to_numpy()
y_raw = data[target].to_numpy()

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

print("Membagi data train dan validasi...")

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

del X, y, data

hasil = {}

for nama, bobot in [
    ("LightGBM Biasa", None),
    ("LightGBM Balanced", "balanced")
]:
    print("\nTraining:", nama)

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(encoder.classes_),
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        max_bin=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2,
        class_weight=bobot,
        random_state=42,
        n_jobs=1,
        force_col_wise=True,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    skor = f1_score(
        y_val,
        pred,
        average="macro"
    )

    hasil[nama] = skor

    print("Macro F1:", round(skor, 5))

print("\n========== HASIL PERBANDINGAN ==========")

for nama, skor in hasil.items():
    print(nama, ":", round(skor, 5))

terbaik = max(hasil, key=hasil.get)
print("\nModel terbaik:", terbaik)


In [ ]:

from lightgbm import LGBMClassifier
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score
)
import pandas as pd

# 1. Membuat model LightGBM
model_macro_lgbm = LGBMClassifier(
    objective="multiclass",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    max_bin=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3,
    n_jobs=1,
    force_col_wise=True,
    random_state=42,
    verbosity=-1
)

# 2. Melatih model
print("Sedang melatih LightGBM...")
model_macro_lgbm.fit(X_train, y_train)

# 3. Prediksi validation
print("Membuat prediksi...")
prediksi = model_macro_lgbm.predict(X_val)

# 4. Menghitung metrik
hasil = pd.DataFrame({
    "Metrik": [
        "Macro F1",
        "Weighted F1",
        "Micro F1",
        "Accuracy",
        "Macro Precision",
        "Macro Recall"
    ],
    "Skor": [
        f1_score(y_val, prediksi, average="macro"),
        f1_score(y_val, prediksi, average="weighted"),
        f1_score(y_val, prediksi, average="micro"),
        accuracy_score(y_val, prediksi),
        precision_score(y_val, prediksi, average="macro", zero_division=0),
        recall_score(y_val, prediksi, average="macro", zero_division=0)
    ]
})

# 5. Mengurutkan skor tertinggi
hasil = hasil.sort_values("Skor", ascending=False)

print("\nHASIL EVALUASI LIGHTGBM")
display(hasil)


In [ ]:

# ==========================================
# PERBAIKAN TIPE DATA LIGHTGBM
# ==========================================

cat_cols = []

for c in features:

    # Deteksi semua fitur non-numerik
    if (
        not pd.api.types.is_numeric_dtype(tr[c].dtype)
        and not pd.api.types.is_bool_dtype(tr[c].dtype)
    ):
        cat_cols.append(c)

        train_values = (
            tr[c].fillna("__MISSING__").astype(str)
        )

        val_values = (
            va[c].fillna("__MISSING__").astype(str)
        )

        # Samakan kategori train dan validation
        categories = pd.Index(
            pd.concat(
                [train_values, val_values],
                ignore_index=True
            ).unique()
        )

        tr[c] = pd.Categorical(
            train_values,
            categories=categories
        )

        va[c] = pd.Categorical(
            val_values,
            categories=categories
        )

# Perbarui data fitur
Xtr = tr[features]
Xva = va[features]

print("Jumlah fitur:", len(features))
print("Fitur kategorikal:", len(cat_cols))
print("\nPemeriksaan tipe data:")
print(Xtr.dtypes.value_counts())

# Pastikan tidak ada tipe string tersisa
bad_cols = [
    c for c in features
    if not (
        pd.api.types.is_numeric_dtype(Xtr[c].dtype)
        or isinstance(Xtr[c].dtype, pd.CategoricalDtype)
    )
]

print("\nKolom bermasalah:", bad_cols)

assert len(bad_cols) == 0, "Masih ada tipe data yang tidak didukung!"

print("\nSEMUA FITUR SIAP UNTUK LIGHTGBM!")


In [ ]:

# ==========================================
# FEATURE IMPORTANCE LIGHTGBM
# ==========================================

import pandas as pd

importance = pd.DataFrame({
    "Fitur": features,
    "Gain": model_pitri.feature_importance(
        importance_type="gain"
    ),
    "Split": model_pitri.feature_importance(
        importance_type="split"
    )
})

importance = importance.sort_values(
    "Gain",
    ascending=False
)

print("20 FITUR PALING BERPENGARUH")
display(importance.head(20))

print("\n20 FITUR DENGAN GAIN TERENDAH")
display(importance.tail(20))


In [ ]:

# ==========================================
# EKSPERIMEN LIGHTGBM: MODEL LEBIH KOMPLEKS
# Tetap menggunakan seluruh 248 fitur
# ==========================================

import lightgbm as lgb
import numpy as np
from sklearn.metrics import f1_score

params_v2 = params.copy()

params_v2.update({
    "learning_rate": 0.025,
    "num_leaves": 127,
    "max_depth": -1,
    "min_data_in_leaf": 60,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l1": 1.0,
    "lambda_l2": 15.0,
    "max_bin": 63,
    "cat_smooth": 20,
    "num_threads": 1,
    "seed": 42
})

dtrain_v2 = lgb.Dataset(
    Xtr,
    label=ytr,
    categorical_feature=cat_cols,
    free_raw_data=True
)

dvalid_v2 = lgb.Dataset(
    Xva,
    label=yva,
    categorical_feature=cat_cols,
    reference=dtrain_v2,
    free_raw_data=True
)

model_v2 = lgb.train(
    params_v2,
    dtrain_v2,
    num_boost_round=800,
    valid_sets=[dvalid_v2],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

proba_v2 = model_v2.predict(
    Xva,
    num_iteration=model_v2.best_iteration
)

pred_v2 = np.argmax(proba_v2, axis=1)

score_v2 = f1_score(
    yva,
    pred_v2,
    average="macro"
)

# Terapkan bobot terbaik dari eksperimen sebelumnya
pred_v2_weighted = np.argmax(
    proba_v2 * weights_fine,
    axis=1
)

score_v2_weighted = f1_score(
    yva,
    pred_v2_weighted,
    average="macro"
)

print("\nHASIL MODEL V2")
print("Best iteration:", model_v2.best_iteration)
print("Macro F1 awal:", round(score_v2, 5))
print("Macro F1 dengan bobot lama:", round(score_v2_weighted, 5))
print("Target eksperimen sebelumnya: 0.47086")


In [ ]:

# ==========================================
# TUNING CLASS WEIGHT MODEL V2
# ==========================================

import numpy as np
from sklearn.metrics import f1_score

weights_v2 = np.ones(len(classes), dtype=float)

best_v2_score = f1_score(
    yva,
    np.argmax(proba_v2, axis=1),
    average="macro"
)

multipliers = [
    0.65, 0.75, 0.85, 0.95,
    1.05, 1.15, 1.30, 1.50,
    1.75, 2.00
]

print("Macro F1 awal:", round(best_v2_score, 5))

for putaran in range(5):
    berubah = False

    for k in range(len(classes)):
        for mult in multipliers:
            kandidat = weights_v2.copy()
            kandidat[k] *= mult

            pred = np.argmax(
                proba_v2 * kandidat,
                axis=1
            )

            skor = f1_score(
                yva,
                pred,
                average="macro"
            )

            if skor > best_v2_score + 1e-6:
                best_v2_score = skor
                weights_v2 = kandidat.copy()
                berubah = True

    print(
        f"Putaran {putaran + 1}: "
        f"Macro F1 = {best_v2_score:.5f}"
    )

    if not berubah:
        break

print("\nHASIL TUNING MODEL V2")
print("Macro F1:", round(best_v2_score, 5))
print("Bobot:", np.round(weights_v2, 4))

pred_v2_final = np.argmax(
    proba_v2 * weights_v2,
    axis=1
)

from sklearn.metrics import classification_report

print(classification_report(
    yva,
    pred_v2_final,
    target_names=classes,
    digits=4,
    zero_division=0
))


In [ ]:

# ==========================================
# MODEL V3 - LOG TRANSFORMATION
# ==========================================

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import f1_score

# Salin dataset sebelumnya
Xtr_v3 = Xtr.copy()
Xva_v3 = Xva.copy()

# Kandidat fitur yang akan ditransformasi
log_candidates = [
    "rs_cost_sum",
    "rs_cost_max",
    "rs_cost_mean",
    "declared_value",
    "value_per_kg",
    "value_per_l",
    "inv_amt_sum",
    "inv_amt_max",
    "inv_amt_mean",
    "inv_amt_per_inv",
    "pk_base_cost",
    "inv_tax_sum",
    "inv_disc_sum"
]

# Hanya gunakan fitur numerik nonnegatif
log_features = []

for col in log_candidates:
    if col not in Xtr_v3.columns:
        continue

    if (
        pd.api.types.is_numeric_dtype(Xtr_v3[col])
        and Xtr_v3[col].min() >= 0
        and Xva_v3[col].min() >= 0
    ):
        log_features.append(col)

        Xtr_v3["log_" + col] = np.log1p(
            Xtr_v3[col].astype("float32")
        )

        Xva_v3["log_" + col] = np.log1p(
            Xva_v3[col].astype("float32")
        )

print("Fitur log yang dibuat:", log_features)
print("Jumlah fitur V3:", Xtr_v3.shape[1])

# Dataset LightGBM
dtrain_v3 = lgb.Dataset(
    Xtr_v3,
    label=ytr,
    categorical_feature=cat_cols,
    free_raw_data=True
)

dvalid_v3 = lgb.Dataset(
    Xva_v3,
    label=yva,
    categorical_feature=cat_cols,
    reference=dtrain_v3,
    free_raw_data=True
)

# Gunakan parameter V2
params_v3 = params_v2.copy()

model_v3 = lgb.train(
    params_v3,
    dtrain_v3,
    num_boost_round=800,
    valid_sets=[dvalid_v3],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

# Prediksi
proba_v3 = model_v3.predict(
    Xva_v3,
    num_iteration=model_v3.best_iteration
)

pred_v3 = np.argmax(proba_v3, axis=1)

score_v3 = f1_score(
    yva,
    pred_v3,
    average="macro"
)

# Pembanding menggunakan bobot V2
pred_v3_weighted = np.argmax(
    proba_v3 * weights_v2,
    axis=1
)

score_v3_weighted = f1_score(
    yva,
    pred_v3_weighted,
    average="macro"
)

print("\nHASIL MODEL V3")
print("Best iteration:", model_v3.best_iteration)
print("Macro F1 awal:", round(score_v3, 5))
print("Macro F1 dengan bobot V2:", round(score_v3_weighted, 5))
print("Pembanding V2:", 0.47934)


In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent.parent
print("Folder proyek:", ROOT)

for folder in ["models", "results", "data/processed"]:
    lokasi = ROOT / "pitri" / folder
    print(f"\nIsi {lokasi}:")
    if lokasi.exists():
        for f in lokasi.iterdir():
            if f.is_file():
                print(f.name, "-", round(f.stat().st_size / 1024**2, 2), "MB")
    else:
        print("Folder belum ditemukan")

In [5]:
import lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent.parent
MODEL_PATH = ROOT / "pitri" / "models" / "lightgbm_final.txt"

model = lgb.Booster(model_file=str(MODEL_PATH))

print("MODEL BERHASIL DIMUAT")
print("Jumlah iterasi:", model.current_iteration())
print("Jumlah fitur:", model.num_feature())

MODEL BERHASIL DIMUAT
Jumlah iterasi: 250
Jumlah fitur: 192


In [9]:
import joblib
from pathlib import Path

P = ROOT / "pitri" / "models" / "train_v2.pkl"

obj = joblib.load(P)

print("Berhasil dibuka!")
print("Tipe:", type(obj))

Berhasil dibuka!
Tipe: <class 'lightgbm.sklearn.LGBMClassifier'>


In [10]:
print("Tipe model:", type(obj))
print("Jumlah fitur:", obj.n_features_in_)
print("Jumlah kelas:", obj.n_classes_)
print("Nama kelas:", obj.classes_)
print("Jumlah iterasi:", obj.n_estimators_)
print("Best iteration:", obj.best_iteration_)

print("\nParameter model:")
print(obj.get_params())

Tipe model: <class 'lightgbm.sklearn.LGBMClassifier'>
Jumlah fitur: 91
Jumlah kelas: 7
Nama kelas: [0 1 2 3 4 5 6]
Jumlah iterasi: 400
Best iteration: 0

Parameter model:
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.5, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 100, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 400, 'n_jobs': None, 'num_leaves': 63, 'objective': 'multiclass', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 10, 'subsample': 0.8, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'num_class': 7, 'verbose': -1}


In [11]:
print("Nama 91 fitur model:")
for i, nama in enumerate(obj.booster_.feature_name()):
    print(i, nama)

Nama 91 fitur model:
0 weight_kg
1 volume_l
2 declared_value
3 distance_km
4 ship_month
5 ship_dow
6 ship_hour
7 is_cod
8 is_promo
9 channel
10 goods_category
11 dest_region
12 insurance_opt
13 qty_items
14 coupon_count
15 gift_wrap
16 fragility_flag
17 order_to_ship_hours
18 prioritas_pengiriman
19 jumlah_catatan
20 flag_alamat_baru
21 berat_volumetrik
22 kota_asal
23 kota_tujuan
24 skor_prioritas_kirim
25 indeks_rute_sekunder
26 faktor_muatan_internal
27 varian_cuaca_rute
28 indeks_waktu_proses
29 skor_kualitas_cek
30 scan_count
31 scan_gap_mean
32 scan_gap_max
33 scan_manual_rate
34 scan_quality_mean
35 scan_unique_hub
36 scan_unique_event
37 inv_count
38 inv_amount_sum
39 inv_amount_mean
40 inv_discount_sum
41 inv_tax_sum
42 inv_days_to_pay_mean
43 inv_penalty_rate
44 route_n_segments
45 route_jarak_total
46 route_durasi_total
47 route_biaya_total
48 route_jarak_mean
49 route_durasi_mean
50 route_unique_moda
51 vehicle_type
52 experience_years
53 contract_type
54 courier_score
55 s

In [13]:
from pathlib import Path

ROOT = Path.cwd().parent.parent
PITRI = ROOT / "pitri"

for folder in [
    PITRI / "data",
    PITRI / "data" / "processed",
    PITRI / "models",
    ROOT / "ghani" / "cache"
]:
    print("\nFOLDER:", folder)
    if folder.exists():
        for f in folder.iterdir():
            if f.is_file():
                print(f.name, round(f.stat().st_size / 1024**2, 2), "MB")


FOLDER: c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\data
inv_feat.csv 56.14 MB
route_feat.csv 68.57 MB
scan_feat.csv 66.55 MB

FOLDER: c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\data\processed
feat_test.parquet 88.46 MB
feat_train.parquet 304.02 MB

FOLDER: c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\models
classes.pkl 0.0 MB
lightgbm_final.txt 4.32 MB
train_v2.pkl 18.51 MB

FOLDER: c:\Users\Pitri Wulandari\Documents\Datavidia\ghani\cache


In [15]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent.parent

sub = pd.read_csv(ROOT / "pitri/results/submission.csv")

print("Bentuk submission:", sub.shape)
print("Kolom:", sub.columns.tolist())
print(sub.head())

Bentuk submission: (362245, 2)
Kolom: ['shipment_id', 'status_penanganan']
    shipment_id     status_penanganan
0  SHP-5A64RENJ  tertunda_operasional
1  SHP-BUZSJTV9     konfirmasi_alamat
2  SHP-EGUZBJHQ                normal
3  SHP-YWBRSQLZ     konfirmasi_alamat
4  SHP-JS89CKME                normal


In [16]:
import pandas as pd
from collections import Counter

ROOT = Path.cwd().parent.parent
sub = pd.read_csv(ROOT / "pitri" / "results" / "submission.csv")

print("Distribusi prediksi submission:")
print(sub["status_penanganan"].value_counts())
print("\nPersentase:")
print((sub["status_penanganan"].value_counts(normalize=True) * 100).round(2))
print("\nJumlah shipment unik:", sub["shipment_id"].nunique())
print("Ada nilai kosong:", sub.isna().sum().to_dict())

Distribusi prediksi submission:
status_penanganan
normal                  117586
konfirmasi_alamat        81524
tertunda_operasional     62063
gagal_antar_awal         42735
retur_pengirim           27156
rusak_perjalanan         20778
klaim_hilang             10403
Name: count, dtype: int64

Persentase:
status_penanganan
normal                  32.46
konfirmasi_alamat       22.51
tertunda_operasional    17.13
gagal_antar_awal        11.80
retur_pengirim           7.50
rusak_perjalanan         5.74
klaim_hilang             2.87
Name: proportion, dtype: float64

Jumlah shipment unik: 362245
Ada nilai kosong: {'shipment_id': 0, 'status_penanganan': 0}


In [17]:
import shutil

shutil.copy2(
    ROOT / "pitri" / "results" / "submission.csv",
    ROOT / "pitri" / "results" / "submission_backup.csv"
)

print("Submission backup berhasil disimpan!")

Submission backup berhasil disimpan!


In [24]:
from pathlib import Path

BASE = Path(
    r"C:\Users\Pitri Wulandari\Documents\Datavidia\pitri"
)

print("FILE DATA PROCESSED:")
for f in (BASE / "data" / "processed").glob("*"):
    print(f.name, "|", round(f.stat().st_size / 1024**2, 2), "MB")

print("\nFILE DATA UTAMA:")
for f in (BASE / "data").glob("*"):
    print(f.name)

FILE DATA PROCESSED:
feat_test.parquet | 88.46 MB
feat_train.parquet | 304.02 MB

FILE DATA UTAMA:
inv_feat.csv
processed
route_feat.csv
scan_feat.csv


In [ ]:

# ============================================================
# DATAVIDIA - MODEL KLASIFIKASI PITRI
# LIGHTGBM + 5 FOLD OOF TARGET ENCODING
# ============================================================

from pathlib import Path
import os
import gc
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")

# ============================================================
# 1. KONFIGURASI FOLDER
# ============================================================

def cari_folder_pitri():
    lokasi = Path.cwd().resolve()

    for p in [lokasi] + list(lokasi.parents):
        if (p / "pitri" / "data" / "processed" / "feat_train.parquet").exists():
            return p / "pitri"

        if (p / "data" / "processed" / "feat_train.parquet").exists():
            return p

    raise FileNotFoundError(
        "Folder pitri tidak ditemukan. "
        "Pastikan notebook berada di folder Datavidia atau pitri."
    )

BASE = cari_folder_pitri()

DATA = BASE / "data" / "processed"
MODEL_DIR = BASE / "models"
RESULT_DIR = BASE / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA / "feat_train.parquet"
TEST_PATH = DATA / "feat_test.parquet"

TARGET = "status_penanganan"
ID_COL = "shipment_id"

SEED = 42
N_FOLDS = 5
SMOOTHING = 20.0

# Percobaan awal: 450 ribu baris.
# Setelah berhasil, ubah menjadi 1050000.
SUB_N = int(os.environ.get("SUB_N", "450000"))

ROUNDS = int(os.environ.get("ROUNDS", "800"))
LEARNING_RATE = float(os.environ.get("LR", "0.05"))

print("Folder proyek:", BASE)
print("Data:", DATA)
print("Jumlah training maksimum:", SUB_N)

# ============================================================
# 2. LOAD DATA
# ============================================================

print("\nMembaca dataset...")

train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

print("Train:", train.shape)
print("Test :", test.shape)

if TARGET not in train.columns:
    raise ValueError("Kolom target tidak ditemukan.")

if ID_COL not in test.columns:
    raise ValueError("shipment_id tidak ditemukan di data test.")

test_ids = test[ID_COL].copy()
y_raw = train[TARGET].astype(str)

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

def buat_fitur(df):
    df = df.copy()

    # Route
    if {"kota_asal", "kota_tujuan"}.issubset(df.columns):
        df["route"] = (
            df["kota_asal"].fillna("UNKNOWN").astype(str)
            + ">"
            + df["kota_tujuan"].fillna("UNKNOWN").astype(str)
        )

    # Customer + kategori barang
    if {"customer_id", "goods_category"}.issubset(df.columns):
        df["cu_route"] = (
            df["customer_id"].fillna("UNKNOWN").astype(str)
            + "|"
            + df["goods_category"].fillna("UNKNOWN").astype(str)
        )

    # Jumlah event scan
    scan_cols = [
        c for c in df.columns
        if "scan" in c.lower()
        and any(k in c.lower() for k in ["count", "total", "_n", "event"])
    ]

    if scan_cols:
        df["sc_n"] = (
            df[scan_cols]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0)
            .sum(axis=1)
        )

    # Gap waktu
    gap_cols = [c for c in df.columns if "gap_" in c.lower()]

    for c in gap_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Rata-rata fitur q
    if "q_mean" not in df.columns:
        q_cols = [
            c for c in df.columns
            if c.lower().startswith("q_")
            and pd.api.types.is_numeric_dtype(df[c])
        ]

        if q_cols:
            df["q_mean"] = df[q_cols].mean(axis=1)

    return df


print("\nMembuat fitur...")

train = buat_fitur(train)
test = buat_fitur(test)

# ============================================================
# 4. SAMPLING STRATIFIED
# ============================================================

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

classes = encoder.classes_
n_classes = len(classes)

print("\nKelas:", list(classes))
print("Jumlah kelas:", n_classes)

if SUB_N < len(train):
    rng = np.random.default_rng(SEED)
    selected = []

    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        jumlah = max(1, round(SUB_N * len(idx) / len(y)))
        jumlah = min(jumlah, len(idx))
        selected.extend(
            rng.choice(idx, jumlah, replace=False)
        )

    selected = np.array(selected)
    rng.shuffle(selected)

    train = train.iloc[selected].reset_index(drop=True)
    y = y[selected]

print("Data training digunakan:", train.shape)

# Simpan kolom mentah untuk target encoding.
# Hanya kolom yang diperlukan yang disimpan.
TE_COLUMNS = [
    "customer_id",
    "courier_id",
    "hub_id",
    "package_id",
    "promo_id",
    "kota_asal",
    "kota_tujuan",
    "route",
    "cu_route"
]

TE_COLUMNS = [
    c for c in TE_COLUMNS
    if c in train.columns and c in test.columns
]

raw_train = train[TE_COLUMNS].copy()
raw_test = test[TE_COLUMNS].copy()

# ============================================================
# 5. SIAPKAN FITUR DASAR
# ============================================================

drop_cols = [TARGET, ID_COL]

feature_cols = [
    c for c in train.columns
    if c not in drop_cols and c in test.columns
]

def encode_features(df, columns):
    result = {}

    for col in columns:
        s = df[col]

        if pd.api.types.is_numeric_dtype(s):
            result[col] = pd.to_numeric(
                s, errors="coerce"
            ).replace(
                [np.inf, -np.inf], np.nan
            ).astype("float32")

        else:
            values = s.fillna("__MISSING__").astype(str)

            hashed = pd.util.hash_pandas_object(
                values, index=False
            ).to_numpy(dtype=np.uint64)

            result[col] = (
                hashed % np.uint64(2147483647)
            ).astype("float32")

    return pd.DataFrame(result, index=df.index)


print("\nMengubah fitur menjadi numerik...")

X = encode_features(train, feature_cols)
X_test = encode_features(test, feature_cols)

del train, test
gc.collect()

# ============================================================
# 6. TARGET ENCODING
# ============================================================

def buat_mapping(keys, labels):
    jumlah_kode = int(keys.max()) + 1

    prior = (
        np.bincount(labels, minlength=n_classes)
        .astype("float32") / len(labels)
    )

    counts = np.bincount(
        keys, minlength=jumlah_kode
    ).astype("float32")

    sums = np.zeros(
        (jumlah_kode, n_classes),
        dtype="float32"
    )

    for cls in range(n_classes):
        sums[:, cls] = np.bincount(
            keys,
            weights=(labels == cls).astype("float32"),
            minlength=jumlah_kode
        )

    encoded = (
        sums + SMOOTHING * prior
    ) / (
        counts[:, None] + SMOOTHING
    )

    return encoded, counts, prior


def encode_te_train_valid(
    train_values,
    valid_values,
    labels
):
    combined = pd.concat(
        [
            train_values.fillna("__MISSING__").astype(str),
            valid_values.fillna("__MISSING__").astype(str)
        ],
        ignore_index=True
    )

    codes, _ = pd.factorize(combined, sort=False)

    train_codes = codes[:len(train_values)]
    valid_codes = codes[len(train_values):]

    mapping, counts, prior = buat_mapping(
        train_codes, labels
    )

    train_te = mapping[train_codes]
    valid_te = np.tile(
        prior, (len(valid_codes), 1)
    ).astype("float32")

    known = valid_codes < len(mapping)

    valid_te[known] = mapping[valid_codes[known]]

    train_count = counts[train_codes]
    valid_count = np.zeros(len(valid_codes), dtype="float32")
    valid_count[known] = counts[valid_codes[known]]

    return train_te, valid_te, train_count, valid_count


def encode_te_final(train_values, test_values, labels):
    combined = pd.concat(
        [
            train_values.fillna("__MISSING__").astype(str),
            test_values.fillna("__MISSING__").astype(str)
        ],
        ignore_index=True
    )

    codes, _ = pd.factorize(combined, sort=False)

    train_codes = codes[:len(train_values)]
    test_codes = codes[len(train_values):]

    mapping, counts, prior = buat_mapping(
        train_codes, labels
    )

    test_te = np.tile(
        prior, (len(test_codes), 1)
    ).astype("float32")

    known = test_codes < len(mapping)
    test_te[known] = mapping[test_codes[known]]

    test_count = np.zeros(len(test_codes), dtype="float32")
    test_count[known] = counts[test_codes[known]]

    return test_te, test_count


# ============================================================
# 7. 5 FOLD STRATIFIED CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

oof = np.zeros(
    (len(y), n_classes),
    dtype="float32"
)

test_predictions = np.zeros(
    (len(X_test), n_classes),
    dtype="float32"
)

fold_scores = []
best_iterations = []

params = {
    "objective": "multiclass",
    "num_class": n_classes,
    "metric": "multi_logloss",
    "learning_rate": LEARNING_RATE,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 100,
    "feature_fraction": 0.50,
    "bagging_fraction": 0.80,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 10.0,
    "max_bin": 63,
    "class_weight": "balanced",
    "verbosity": -1,
    "num_threads": 2,
    "random_state": SEED
}

models = []

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(X, y), start=1
):
    print(f"\n========== FOLD {fold}/{N_FOLDS} ==========")

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()

    # Target encoding dihitung hanya dari fold training
    for col in TE_COLUMNS:
        tr_te, va_te, tr_count, va_count = encode_te_train_valid(
            raw_train[col].iloc[tr_idx],
            raw_train[col].iloc[va_idx],
            y[tr_idx]
        )

        for cls in range(n_classes):
            name = f"te_{col}_{cls}"
            X_tr[name] = tr_te[:, cls]
            X_va[name] = va_te[:, cls]

        X_tr[f"cnt_{col}"] = tr_count
        X_va[f"cnt_{col}"] = va_count

        del tr_te, va_te
        gc.collect()

    model = lgb.LGBMClassifier(
        **params,
        n_estimators=ROUNDS
    )

    model.fit(
        X_tr,
        y[tr_idx],
        eval_set=[(X_va, y[va_idx])],
        callbacks=[
            lgb.early_stopping(60, verbose=False),
            lgb.log_evaluation(100)
        ]
    )

    valid_proba = model.predict_proba(X_va)
    oof[va_idx] = valid_proba

    score = f1_score(
        y[va_idx],
        valid_proba.argmax(axis=1),
        average="macro"
    )

    fold_scores.append(score)
    best_iterations.append(model.best_iteration_)

    print(f"Fold {fold} Macro F1: {score:.6f}")
    print("Best iteration:", model.best_iteration_)

    # Target encoding test memakai seluruh fold training.
    X_te_fold = X_test.copy()

    for col in TE_COLUMNS:
        te_values, te_count = encode_te_final(
            raw_train[col].iloc[tr_idx],
            raw_test[col],
            y[tr_idx]
        )

        for cls in range(n_classes):
            X_te_fold[f"te_{col}_{cls}"] = te_values[:, cls]

        X_te_fold[f"cnt_{col}"] = te_count

        del te_values, te_count
        gc.collect()

    # Samakan susunan kolom
    X_te_fold = X_te_fold.reindex(
        columns=X_tr.columns,
        fill_value=0
    )

    test_predictions += (
        model.predict_proba(X_te_fold) / N_FOLDS
    )

    models.append(model)

    # Simpan model tiap fold
    model.booster_.save_model(
        str(MODEL_DIR / f"lightgbm_fold_{fold}.txt")
    )

    del X_tr, X_va, X_te_fold, valid_proba
    gc.collect()

# ============================================================
# 8. EVALUASI OOF
# ============================================================

oof_score = f1_score(
    y,
    oof.argmax(axis=1),
    average="macro"
)

print("\n====================================")
print("HASIL CROSS VALIDATION PITRI")
print("====================================")

for i, score in enumerate(fold_scores, start=1):
    print(f"Fold {i}: {score:.6f}")

print("Rata-rata fold:", round(float(np.mean(fold_scores)), 6))
print("OOF Macro F1  :", round(float(oof_score), 6))

print("\nClassification report:")
print(
    classification_report(
        y,
        oof.argmax(axis=1),
        target_names=classes,
        zero_division=0
    )
)

# ============================================================
# 9. TUNING BOBOT KELAS DARI OOF
# ============================================================

weights = np.ones(n_classes, dtype="float32")
best_score = oof_score

for step in [0.20, 0.10, 0.05]:
    improved = True

    while improved:
        improved = False

        for cls in range(n_classes):
            for change in [1 - step, 1 + step]:
                candidate = weights.copy()
                candidate[cls] *= change

                pred = (
                    oof * candidate
                ).argmax(axis=1)

                score = f1_score(
                    y,
                    pred,
                    average="macro"
                )

                if score > best_score:
                    weights = candidate
                    best_score = score
                    improved = True

print("\nMacro F1 setelah tuning:", round(float(best_score), 6))
print("Bobot kelas:", weights.tolist())

# ============================================================
# 10. PREDIKSI TEST DAN SUBMISSION
# ============================================================

print("\nMembuat submission...")

test_pred = (
    test_predictions * weights
).argmax(axis=1)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: encoder.inverse_transform(test_pred)
})

# Validasi hasil
assert len(submission) == len(test_ids)
assert submission[ID_COL].is_unique
assert submission[TARGET].notna().all()
assert submission[TARGET].isin(classes).all()

output = RESULT_DIR / "submission.csv"
submission.to_csv(output, index=False)

# ============================================================
# 11. SIMPAN MODEL DAN METADATA
# ============================================================

with open(MODEL_DIR / "classes.pkl", "wb") as f:
    pickle.dump(
        {
            "classes": classes.tolist(),
            "features": feature_cols,
            "te_columns": TE_COLUMNS,
            "weights": weights.tolist(),
            "best_iterations": best_iterations
        },
        f
    )

with open(RESULT_DIR / "meta.json", "w") as f:
    json.dump(
        {
            "model": "LightGBM Multiclass 5-Fold",
            "rows_used": int(len(y)),
            "test_rows": int(len(test_ids)),
            "fold_scores": [float(s) for s in fold_scores],
            "mean_fold_macro_f1": float(np.mean(fold_scores)),
            "oof_macro_f1": float(oof_score),
            "tuned_oof_macro_f1": float(best_score),
            "best_iterations": [int(i) for i in best_iterations],
            "classes": classes.tolist(),
            "weights": weights.tolist(),
            "features": feature_cols,
            "target_encoding": TE_COLUMNS
        },
        f,
        indent=4
    )

print("\n====================================")
print("TRAINING PITRI SELESAI")
print("====================================")
print("OOF Macro F1 :", round(float(oof_score), 6))
print("Tuned F1     :", round(float(best_score), 6))
print("Submission   :", output)
print("Model        :", MODEL_DIR)
print("Jumlah baris :", len(submission))
print("====================================")

display(submission.head())


NameError: name '__file__' is not defined